In [0]:
from pyspark.sql import functions as F

spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA default")

PREFIX = "wdatt_movie_"


In [0]:
silver_movie_master = spark.table(f"{PREFIX}silver_movie_master")
silver_ratings      = spark.table(f"{PREFIX}silver_ratings")
silver_scraped      = spark.table(f"{PREFIX}silver_scraped_metadata")


In [0]:
top500_ids = silver_scraped.select("movieId").distinct()
print("top500 movieIds:", top500_ids.count())


In [0]:
gold_dim_movies_enriched = (
    silver_movie_master
    .join(top500_ids, on="movieId", how="inner")
    .select(
        "movieId","title_clean","movie_year","genres_array",
        "imdbId","tmdbId",
        "director","budget_clean","poster_url","scraped_title","status"
    )
)

gold_dim_movies_enriched.write.format("delta").mode("overwrite").saveAsTable(f"{PREFIX}gold_dim_movies_enriched")
print("gold_dim_movies_enriched:", gold_dim_movies_enriched.count())


In [0]:
gold_fact_ratings = (
    silver_ratings
    .join(top500_ids, on="movieId", how="inner")
    .select("userId","movieId","rating","timestamp","rating_ts")
)

gold_fact_ratings.write.format("delta").mode("overwrite").partitionBy("movieId").saveAsTable(f"{PREFIX}gold_fact_ratings")
print("gold_fact_ratings:", spark.table(f"{PREFIX}gold_fact_ratings").count())


In [0]:
gold_dim_users = (
    spark.table(f"{PREFIX}gold_fact_ratings")
    .groupBy("userId")
    .agg(
        F.count("*").alias("num_ratings"),
        F.avg("rating").alias("avg_rating"),
        F.min("rating_ts").alias("first_rating_ts"),
        F.max("rating_ts").alias("last_rating_ts"),
        F.countDistinct("movieId").alias("distinct_movies_rated")
    )
    .withColumn("is_power_user", F.col("num_ratings") >= F.lit(50))
)

gold_dim_users.write.format("delta").mode("overwrite").saveAsTable(f"{PREFIX}gold_dim_users")
print("gold_dim_users:", gold_dim_users.count())


In [0]:
assert spark.table(f"{PREFIX}gold_fact_ratings").count() > 0
assert spark.table(f"{PREFIX}gold_dim_movies_enriched").count() == 500
assert spark.table(f"{PREFIX}gold_dim_users").count() > 0
print("✅ GOLD validated")
